# 04 — Smart Metering

Materialises the **shared dimensions** and the curated meter profile described in
[`../specifications/04-smart-metering.md`](../specifications/04-smart-metering.md).
**Run this notebook first** — capabilities 01–03 read its outputs.

**Shared dimensions (owned here)**
- `volume_forecast_dim_zones`
- `volume_forecast_dim_segments`
- `volume_forecast_dim_intervals`

**Capability tables**
- `volume_forecast_bronze_meter_reads`, `volume_forecast_bronze_meter_events`
- `volume_forecast_silver_btm_pv` (behind-the-meter PV deducted from gross)
- `volume_forecast_silver_meter_profile` (curated net load profile — shared input to 01–03)
- `volume_forecast_gold_metering_quality` (completeness / lateness / corrupt SLA)

**Note:** `main.md` frames metering as high-cardinality Auto Loader / streaming ingest. For a reproducible
demo this notebook **materialises the same gold in batch** — a small sample of synthetic meters plus
aggregated profiles over a fixed set of 15-minute intervals. Individual meters are never exposed below
the segment/zone aggregate.

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")

spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}` COMMENT 'Energy trading demo data'")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


random.seed(11)

TODAY = dt.date.today()
# Rolling horizon: 2 settled history days, today (half settled), 2 forecast days ahead.
DELIVERY_DATES = [TODAY + dt.timedelta(days=d) for d in (-2, -1, 0, 1, 2)]
TODAY_INDEX = DELIVERY_DATES.index(TODAY)
LATEST_DATE = DELIVERY_DATES[-1]
N_INTERVALS = 96
NOW_INDEX = 56  # ~14:00 "now" cursor on TODAY; earlier-on-today and all prior days = settled
ZONES = ["DE", "NL", "FR", "BE", "AT"]

# A small fixed set of European public holidays so day_type can legitimately be HOLIDAY
# when a delivery date lands on one (sparse, like the real world).
HOLIDAYS = {(1, 1), (5, 1), (10, 3), (12, 25), (12, 26)}


def interval_ts(day: dt.date, idx: int) -> dt.datetime:
    return dt.datetime.combine(day, dt.time(0, 0)) + dt.timedelta(minutes=15 * idx)


def is_settled(day: dt.date, idx: int) -> bool:
    # Settled = realised. All days before today, plus today up to the NOW cursor.
    if day < TODAY:
        return True
    if day == TODAY:
        return idx < NOW_INDEX
    return False


def season_of(day: dt.date) -> str:
    m = day.month
    if m in (12, 1, 2):
        return "WINTER"
    if m in (3, 4, 5):
        return "SPRING"
    if m in (6, 7, 8):
        return "SUMMER"
    return "AUTUMN"


def day_type_of(day: dt.date) -> str:
    if (day.month, day.day) in HOLIDAYS:
        return "HOLIDAY"
    return "WEEKEND" if day.weekday() >= 5 else "WEEKDAY"


# ---- Shared dimension: zones ----
zones = [
    Row(zone_code="DE", country="Germany",     tso_name="50Hertz/Amprion/TenneT/TransnetBW", control_area="DE-LU", eic_code="10Y1001A1001A82H", currency="EUR"),
    Row(zone_code="NL", country="Netherlands", tso_name="TenneT NL", control_area="NL", eic_code="10YNL----------L", currency="EUR"),
    Row(zone_code="FR", country="France",      tso_name="RTE",       control_area="FR", eic_code="10YFR-RTE------C", currency="EUR"),
    Row(zone_code="BE", country="Belgium",     tso_name="Elia",      control_area="BE", eic_code="10YBE----------2", currency="EUR"),
    Row(zone_code="AT", country="Austria",     tso_name="APG",       control_area="AT", eic_code="10YAT-APG------L", currency="EUR"),
]
spark.createDataFrame(zones).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_dim_zones"))

# ---- Shared dimension: customer segments ----
segments = [
    Row(segment_code="RES",    segment_name="Residential",                  customer_class="RESIDENTIAL", has_btm_pv=False, has_ev=False, default_tariff="RES_FLAT"),
    Row(segment_code="RES_PV", segment_name="Residential with rooftop PV",  customer_class="RESIDENTIAL", has_btm_pv=True,  has_ev=False, default_tariff="RES_PV"),
    Row(segment_code="RES_EV", segment_name="Residential with EV charging", customer_class="RESIDENTIAL", has_btm_pv=False, has_ev=True,  default_tariff="RES_EV_TOU"),
    Row(segment_code="SME",    segment_name="Small & medium enterprise",    customer_class="COMMERCIAL",  has_btm_pv=False, has_ev=False, default_tariff="SME_PROFILE"),
    Row(segment_code="CNI",    segment_name="Commercial & industrial",      customer_class="INDUSTRIAL",  has_btm_pv=False, has_ev=False, default_tariff="CNI_PROFILE"),
]
spark.createDataFrame(segments).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_dim_segments"))

# ---- Shared dimension: 15-minute time spine ----
intervals = []
for day in DELIVERY_DATES:
    dtype = day_type_of(day)
    seas = season_of(day)
    for idx in range(N_INTERVALS):
        start = interval_ts(day, idx)
        intervals.append(Row(
            delivery_date=day, interval_start=start, interval_end=start + dt.timedelta(minutes=15),
            interval_index=idx + 1, hour=start.hour, day_type=dtype, season=seas,
            is_peak=(8 <= start.hour < 20), solar_window=(10 <= start.hour < 16),
        ))
spark.createDataFrame(intervals).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_dim_intervals"))

print("zones:", spark.table(fq("volume_forecast_dim_zones")).count())
print("segments:", spark.table(fq("volume_forecast_dim_segments")).count())
print("intervals:", spark.table(fq("volume_forecast_dim_intervals")).count())

In [ ]:
# Load model + bronze meter reads/events + silver BTM-PV and curated profile.
# Per-segment reference peak MW (DE basis); other zones scaled by ZONE_SCALE.
SEG_PEAK_MW = {"RES": 120.0, "RES_PV": 60.0, "RES_EV": 40.0, "SME": 90.0, "CNI": 150.0}
SEG_BTM_PV_MW = {"RES": 6.0, "RES_PV": 70.0, "RES_EV": 5.0, "SME": 20.0, "CNI": 10.0}
ZONE_SCALE = {"DE": 1.0, "NL": 0.7, "FR": 0.9, "BE": 0.5, "AT": 0.4}
SEG_METERS = {"RES": 480000, "RES_PV": 120000, "RES_EV": 90000, "SME": 40000, "CNI": 6000}
CURATE_TS = dt.datetime.combine(LATEST_DATE, dt.time(15, 0))


def solar_factor(idx: int) -> float:
    h = idx * 15 / 60.0
    if h <= 6 or h >= 19:
        return 0.0
    return max(0.0, math.sin(math.pi * (h - 6.0) / 13.0))


def load_factor(idx: int, segment: str, day_type: str = "WEEKDAY") -> float:
    h = idx * 15 / 60.0
    off = day_type in ("WEEKEND", "HOLIDAY")  # days off reshape demand
    if segment in ("RES", "RES_PV", "RES_EV"):
        m_center = 9.5 if off else 7.5        # later wake-up on days off
        m_amp = 0.42 if off else 0.55
        morning = m_amp * math.exp(-((h - m_center) ** 2) / 3.0)
        evening = 0.55 * math.exp(-((h - 19.5) ** 2) / 4.0)
        midday = (0.15 if off else 0.0) * math.exp(-((h - 13.0) ** 2) / 8.0)  # more at-home daytime use
        base = 0.45 + max(morning, evening) + midday
        if segment == "RES_EV":
            base += 0.35 * math.exp(-((h - 22.0) ** 2) / 3.0)  # overnight EV charging bump
    elif segment == "SME":
        base = (0.30 + 0.20 * math.exp(-((h - 13.0) ** 2) / 16.0)) if off else (0.35 + 0.55 * math.exp(-((h - 13.0) ** 2) / 16.0))
    else:  # CNI continuous load; modest weekend/holiday slowdown
        base = 0.82 + 0.06 * math.sin(h / 24.0 * 2 * math.pi)
        if off:
            base *= 0.85
    return max(0.05, min(1.25, base))


# ---- Silver: behind-the-meter PV + curated net load profile (all dates) ----
btm_rows, profile_rows = [], []
for day in DELIVERY_DATES:
    seas = season_of(day)
    dtype = day_type_of(day)
    seasonal = 1.15 if seas == "WINTER" else (0.95 if seas == "SUMMER" else 1.0)
    for z in ZONES:
        scale = ZONE_SCALE[z]
        for s in segments:
            seg = s.segment_code
            peak = SEG_PEAK_MW[seg] * scale * seasonal
            pv_cap = SEG_BTM_PV_MW[seg] * scale
            n_meters = int(SEG_METERS[seg] * scale)
            for idx in range(N_INTERVALS):
                gross = peak * load_factor(idx, seg, dtype) * (1 + random.gauss(0, 0.02))
                btm = pv_cap * solar_factor(idx) * (1 + random.gauss(0, 0.05))
                btm = max(0.0, btm if (s.has_btm_pv or pv_cap > 0) else 0.0)
                net = max(0.0, gross - btm)
                r = random.random()
                dq = "SUSPECT" if r < 0.01 else ("IMPUTED" if r < 0.04 else "GOOD")
                start = interval_ts(day, idx)
                if btm > 0:
                    btm_rows.append(Row(
                        delivery_date=day, interval_start=start, zone_code=z, segment_code=seg,
                        btm_pv_mw=round(btm, 3),
                        n_pv_meters=int(n_meters * (0.8 if s.has_btm_pv else 0.05)),
                    ))
                profile_rows.append(Row(
                    delivery_date=day, interval_start=start, zone_code=z, segment_code=seg,
                    gross_load_mw=round(gross, 3), btm_pv_mw=round(btm, 3), net_load_mw=round(net, 3),
                    n_meters=n_meters, dq_status=dq, profile_version=1, as_of_ts=CURATE_TS,
                ))
spark.createDataFrame(btm_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_silver_btm_pv"))
spark.createDataFrame(profile_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_silver_meter_profile"))

# ---- Bronze: a small sample of raw meter reads + lifecycle events (today, settled intervals) ----
read_rows, event_rows = [], []
SAMPLE_DAY = TODAY
sample_dtype = day_type_of(SAMPLE_DAY)
ingest_ts = dt.datetime.combine(SAMPLE_DAY, dt.time(0, 5))
SAMPLE_METERS = 6
for z in ZONES:
    for s in segments:
        seg = s.segment_code
        n_meters = max(1, int(SEG_METERS[seg] * ZONE_SCALE[z]))
        per_meter_peak_kw = SEG_PEAK_MW[seg] * ZONE_SCALE[z] * 1000.0 / n_meters
        pv_per_meter_kw = (SEG_BTM_PV_MW[seg] * ZONE_SCALE[z] * 1000.0 / n_meters) if s.has_btm_pv else 0.0
        for m in range(SAMPLE_METERS):
            mid = f"MTR_{z}_{seg}_{m:04d}"
            for idx in range(N_INTERVALS):
                if not is_settled(SAMPLE_DAY, idx):
                    continue
                cons_kw = per_meter_peak_kw * load_factor(idx, seg, sample_dtype) * (1 + random.gauss(0, 0.15))
                exp_kw = pv_per_meter_kw * solar_factor(idx) * (1 + random.gauss(0, 0.1))
                q = random.random()
                if q < 0.01:
                    quality, cons_kw = "CORRUPT", 0.0
                elif q < 0.03:
                    quality = "ESTIMATED"
                elif q < 0.04:
                    quality, cons_kw = "MISSING", 0.0
                else:
                    quality = "OK"
                read_rows.append(Row(
                    ingestion_ts=ingest_ts, read_ts=interval_ts(SAMPLE_DAY, idx), meter_id=mid,
                    segment_code=seg, zone_code=z,
                    consumption_kwh=round(max(0.0, cons_kw) * 0.25, 4),
                    export_kwh=round(max(0.0, exp_kw) * 0.25, 4), read_quality=quality,
                ))
            if random.random() < 0.3:
                event_rows.append(Row(
                    ingestion_ts=ingest_ts, event_ts=interval_ts(SAMPLE_DAY, random.randint(0, NOW_INDEX)),
                    meter_id=mid, event_type=random.choice(["GAP", "LATE", "TAMPER", "RECONNECT", "FW_UPDATE"]),
                    detail="synthetic meter event",
                ))
spark.createDataFrame(read_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_meter_reads"))
spark.createDataFrame(event_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_meter_events"))

print("btm_pv:", spark.table(fq("volume_forecast_silver_btm_pv")).count())
print("meter_profile:", spark.table(fq("volume_forecast_silver_meter_profile")).count())
print("meter_reads:", spark.table(fq("volume_forecast_bronze_meter_reads")).count())
print("meter_events:", spark.table(fq("volume_forecast_bronze_meter_events")).count())

In [ ]:
# Gold: metering data-quality SLA rollup (per day / zone / segment).
quality_rows = []
for day in DELIVERY_DATES:
    for z in ZONES:
        for s in segments:
            n_meters = max(1, int(SEG_METERS[s.segment_code] * ZONE_SCALE[z]))
            expected = n_meters * N_INTERVALS
            completeness = round(random.uniform(97.0, 99.95), 2)
            late = round(random.uniform(0.3, 3.0), 2)
            corrupt = round(random.uniform(0.1, 1.6), 2)
            received = int(expected * completeness / 100.0)
            imputed = int(N_INTERVALS * (corrupt / 100.0) * 2)
            if completeness >= 98.0 and corrupt <= 1.0:
                status = "GOOD"
            elif completeness < 97.5 or corrupt > 1.4:
                status = "BREACH"
            else:
                status = "WATCH"
            quality_rows.append(Row(
                delivery_date=day, zone_code=z, segment_code=s.segment_code,
                expected_reads=expected, received_reads=received, completeness_pct=completeness,
                late_read_pct=late, corrupt_read_pct=corrupt, imputed_intervals=imputed, dq_status=status,
            ))
spark.createDataFrame(quality_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_metering_quality"))

display(spark.table(fq("volume_forecast_gold_metering_quality")).orderBy(F.col("delivery_date").desc(), "zone_code", "segment_code"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_dim_zones",
    "volume_forecast_dim_segments",
    "volume_forecast_dim_intervals",
    "volume_forecast_bronze_meter_reads",
    "volume_forecast_bronze_meter_events",
    "volume_forecast_silver_btm_pv",
    "volume_forecast_silver_meter_profile",
    "volume_forecast_gold_metering_quality",
]:
    print(f"  {t:44s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_04_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 04.")